# Predicting on new dataset (Inference)

## Creating the classes and model structure

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd

# Transformations
from torchvision.tv_tensors import BoundingBoxes
import torchvision.transforms.v2 as T

# Model
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead

In [ ]:
class ObjectDetectionDataset(Dataset):
    def __init__(self, csv_file, transform=None, image_subset = None):
        self.base_path = "data/images_train/" #
        self.annotations = pd.read_csv(csv_file)
        self.transform = transform

        # Group annotations by image as there are 0-n boxes per image
        self.image_groups = self.annotations.groupby("filename")

        # Unique image paths
        # NOTE: use only pre-defined-subset of the image-paths; this enables the use of train-eval split
        all_image_paths = list(self.image_groups.groups.keys())
        if image_subset is not None:
            self.image_paths = image_subset
        else:
            self.image_paths = all_image_paths

    def __len__(self): # Returns the length of the IMAGES in dataset, not boxes
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path =  self.image_paths[idx]

        # Note: take all the rows per each image, so there is no leakage between test and train.
        rows = self.image_groups.get_group(image_path) # Get all bboxes for image

        # Load image
        image = Image.open(self.base_path + image_path).convert("RGB")

        # Create lists to store the boxes and their labels
        boxes = []
        labels = []

        for _, row in rows.iterrows():
            boxes.append([
                row["xmin"],
                row["ymin"],
                row["xmax"],
                row["ymax"]
            ])
            labels.append(row["class_num"])

        # Operate based on if boxes on image:
        if len(boxes)==0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0, ), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        # Create target:
        target = {"boxes": boxes, "labels": labels}

        # Convert to BoundingBox format to use the box-aware transforms.v2 -library
        target["boxes"] = BoundingBoxes(target["boxes"],
                                        format = "XYXY",
                                        canvas_size=image.size[::1]
                                        )

        if self.transform:
            image, target = self.transform(image, target)

        return image, target

In [ ]:
# validation transforms; this only converts images to tensor, no augmentations
val_transforms = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
val_dataset = ObjectDetectionDataset(
    csv_file="data/train_256px_annotations_extended_cleaned_2026.csv",
    transform = val_transforms
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    # pretrained=True, # Deprecated param
    weights=None,
    min_size=256,
    max_size=256
)

# Adjust the number of classes:
num_classes = 3  # background + 2 object classes
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# NOTE: Probably not needed for prediction?
# ROI-head customization:
# 1) Take much smaller sample; it only takes whatever positive samples (targets) are in image, then fills the rest with negatives!! Medium number of bboxs per image is 4, max is 40
# 2) Positive fraction is the MAX fraction to take in
# 3) post_nms_top_n is the overall top-k proposals passed on to the ROI-head (from RPN); this can also be limited from default 2000.
model.roi_heads.fg_bg_sampler.batch_size_per_image = 64 # Expecting 4:60 ratio on avg, i.e. 6.25%. (was 4:508, i.e. 0.8%)
model.roi_heads.fg_bg_sampler.positive_fraction = 0.5
# Limit the overall number of proposals to smaller number
model.rpn._post_nms_top_n['training'] = 256 # send 512 best proposals to roi-head


# Anchor generator (customized version)
anchor_generator = AnchorGenerator(
    sizes=((10,),
           (12,),
           (16,),
           (20,),
           (28,),),  # MUCH smaller than defaults; note that these are taken from EDA
    aspect_ratios=((0.82, 1.0, 1.1),  # NOTE: Added per-size-ratios from eda-notebook: These follow the 20p / 50p / 80p quantiles.
                   (0.83, 0.92, 1.1),
                   (0.82, 0.94, 1.12),
                   (0.81, 0.95, 1.12),
                   (0.93, 1.05, 1.25),)
)

model.rpn.anchor_generator = anchor_generator



# Load model weights from the saved state:
model.load_state_dict(torch.load("models/fbestmodel_fasterrcnn_2026-03-07_v2"))
model.eval()



# Model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model ready on device:", device)


## Pre-process the new data


- 0) Store the metadata of full image (year, geographic coords of corners)
- 1) Clip to 256x256 px; use stride of 128 pixels to ensure no part is falling on the edge of clipped image
- 2) Store the starting location of the clipped images relative to original images
- 3) Use the basic transform


In [ ]:
# TODO: Read one master-image, clip, transform, bundle to batches and send to device

In [ ]:
# TODO: Wrap below loading and pre-processing into function

image_path = "test_images/image1.png"

image = Image.open(image_path).convert("RGB")
image_tensor = val_transforms(image) # USE: val_transforms

image_tensor = image_tensor.to(device)




## Predict


- 4) Loop through the modeling
- 5) Merge the results back to one final image using nms-process; take the average of overlapping predictionsfor confidence (and voting for the class)
- 6) Convert the bboxes to geographical coordinates


In [ ]:
# NOTE: Model expects a LIST of images
# --> BUndle multiple images into same list to get faster predictions (batch-size about 64 (?) should fit into gpu memory)
with torch.no_grad():
    predictions = model([image_tensor])

In [ ]:
# Predictions is a list (one peri image)

pred = predictions[0]

boxes = pred["boxes"]
scores = pred["scores"]
labels = pred["labels"]


# FIlter only high-confidence
threshold = 0.7

keep = scores > threshold
boxes = boxes[keep]
scores = scores[keep]
labels = labels[keep]
